In [1]:
import os 
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async" 
import tensorflow as tf 
tf.keras.backend.clear_session() 
import pandas as pd 
from deepmreye import analyse, architecture, preprocess, train 
from deepmreye.util import data_generator, model_opts, util 
import numpy as np 

gpus = tf.config.experimental.list_physical_devices('GPU') 
tf.config.experimental.set_memory_growth(gpus[0], True)

2026-02-04 11:21:07.503709: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-04 11:21:07.580306: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-04 11:21:07.580349: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-04 11:21:07.582517: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-04 11:21:07.598335: I tensorflow/core/platform/cpu_feature_guar

In [6]:
import json
import ray

from ray import tune
from ray.train import report
from ray.tune.schedulers import ASHAScheduler

from smooth_loss import train_model_with_smoothl1


In [2]:
def create_holdout_generators(datasets=None, train_split=0.6, train_list=None, test_list=None, **args):
    # If train_list and test_list are provided, use them directly
    if train_list is not None and test_list is not None:
        full_training_list = train_list
        full_testing_list = test_list
    else:
        # Otherwise, build from datasets
        full_training_list, full_testing_list = list(), list()
        for fn_data in datasets:
            this_file_list = [fn_data + p for p in os.listdir(fn_data)]
            np.random.shuffle(this_file_list)
            split_idx = int(train_split * len(this_file_list))
            this_training_list = this_file_list[0:split_idx]
            this_testing_list = this_file_list[split_idx:]
            full_training_list.extend(this_training_list)
            full_testing_list.extend(this_testing_list)

    # Call create_generators with the final lists
    (
        training_generator,
        testing_generator,
        single_testing_generators,
        single_testing_names,
        single_training_generators,
        single_training_names,
    ) = data_generator.create_generators(full_training_list, full_testing_list, **args)

    return (
        training_generator,
        testing_generator,
        single_testing_generators,
        single_testing_names,
        single_training_generators,
        single_training_names,
        full_testing_list,
        full_training_list,
    )

In [12]:
!rm -rf /tmp/ray/*
!rm -rf ./ray_results/*
!rm -rf ./ray_spill/*

In [3]:
opts = model_opts.get_opts()
print(opts)

{'kernel': 3, 'lr': 2e-05, 'filters': 32, 'multiplier': 2, 'depth': 4, 'dropout_rate': 0.1, 'num_dense': 2, 'num_fc': 1024, 'gaussian_noise': 0, 'activation': <function mish at 0x7fdf1c178040>, 'groups': 8, 'inner_timesteps': 10, 'loss_euclidean': 1, 'loss_confidence': 0.1, 'epochs': 25, 'steps_per_epoch': 1500, 'validation_steps': 1500, 'train_test_split': 0.6, 'batch_size': 8, 'mixed_batches': True, 'mc_dropout': False, 'rotation_x': 5, 'rotation_y': 5, 'rotation_z': 5, 'shift': 4, 'zoom': 0.15}


In [4]:
opts['mc_dropout'] = True
opts['gaussian_noise']=0.05
opts['epochs']=100
opts['loss_confidence'] = 0.5
opts['steps_per_epoch'] = (182 * 18) // 8  # ~410 steps
opts['validation_steps'] = (182 * 6) // 8   # ~136 steps
opts['lr']=5e-5
opts['smooth_l1_delta'] = 1
opts['depth'] = 2  # fewer residual blocks
opts['num_fc'] = 128  # smaller FC layer
opts['dropout_rate'] = 0.3  # more regularization

In [7]:
train_list_rs=np.loadtxt('/mnt/compneuro/deepmreye_finetuning/Zosia/train_list_rs.txt',dtype=str)
test_list_rs=np.loadtxt('/mnt/compneuro/deepmreye_finetuning/Zosia/test_list_rs.txt',dtype=str)

In [14]:
opts["epochs"] = 50
opts["steps_per_epoch"] = 10
opts["validation_steps"] = 4

results_path = "/mnt/compneuro/deepmreye_finetuning/Cemal/ray_results"
spill_path = "/mnt/compneuro/deepmreye_finetuning/Cemal/ray_spill"

if ray.is_initialized():
    ray.shutdown()

os.environ["RAY_DISABLE_METRICS"] = "1"
os.environ["RAY_DISABLE_DASHBOARD"] = "1"

ray.init(
    _system_config={
        "object_spilling_config": json.dumps(
            {"type": "filesystem", "params": {"directory_path": spill_path}}
        )
    },
    ignore_reinit_error=True
)


2026-02-02 08:05:54,831	WARNING node.py:1846 -- The object spilling config is specified from an unstable API - system config or environment variable. This is subject to change in the future. You can use the stable API - --object-spilling-directory in ray start or object_spilling_directory in ray.init() to specify the object spilling directory instead. If you need more advanced settings, please open a github issue with the Ray team.
2026-02-02 08:06:00,843	INFO worker.py:2007 -- Started a local Ray instance.
/home/CemalKoba/venvs/deepmreye-gpu/lib/python3.10/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.10.12
Ray version:,2.53.0


(pid=gcs_server) [2026-02-02 08:06:25,686 E 1084693 1084693] (gcs_server) gcs_server.cc:303: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(raylet) [2026-02-02 08:06:29,857 E 1084866 1084866] (raylet) main.cc:1032: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(pid=1085027) [2026-02-02 08:06:33,361 E 1085027 1085130] core_worker_process.cc:842: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


In [15]:
train_list = train_list_rs[5:]       # 18 samples → training
val_list   = train_list_rs[:5]       # 5 samples → validation
test_list  = test_list_rs            # 6 samples → final test (unused here)


base_opts = opts.copy()
def train_deepmreye_ray(config):
    print("Trial started", config, flush=True)

    try:
        trial_opts = base_opts.copy()
        trial_opts.update(config)

        # GPU
        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            tf.config.experimental.set_memory_growth(gpus[0], True)

        # ✅ TYLKO 23 → train, TYLKO 6 → val
        generators = data_generator.create_generators(
            full_training_list=train_list,   
            full_testing_list=val_list,     
            batch_size=trial_opts["batch_size"]
        )

        generators = (
            *generators,
            train_list,
            val_list
        )

        model, model_inf, history = train_model_with_smoothl1(
            dataset="ray_trial",
            generators=generators,
            opts=trial_opts,
            is_resting_state=True,
            save=False,
            verbose=0,
            use_multiprocessing=False,
            workers=1,
            pretrained_weights="/mnt/compneuro/deepmreye_finetuning/Zosia/modelinference_resting_state_data_18samples.h5",
            freeze_backbone=True   # 🔑 FINE-TUNING
        )

        val_losses = history.history.get("val_smoothl1_loss", [])
        best_loss = min(val_losses)

        # ✅ POPRAWNE REPORTOWANIE
        tune.report({"loss": best_loss})

    except Exception:
        import traceback
        traceback.print_exc()
        tune.report({"loss": float("inf")})


[2026-02-02 08:06:35,626 E 1080489 1084993] core_worker_process.cc:842: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


In [16]:
search_space = {
    # Core sensitivity params
    "lr": tune.loguniform(1e-6, 1e-4),
    "smooth_l1_delta": tune.choice([0.5, 1.0, 1.5, 2.0]),  # 🔥 Match your ±2° range
    
    # Regularization (fight overfitting)
    "dropout_rate": tune.choice([0.2, 0.3, 0.4]),  # Higher for small dataset
    "gaussian_noise": tune.choice([0.0, 0.02, 0.05]),
    
    # Architecture (reduce capacity)
    "depth": tune.choice([2, 3]),           # 🔥 Fewer residual blocks
    "num_fc": tune.choice([128, 256]),      # 🔥 Smaller FC layers
    
    # Loss balance
    "loss_confidence": tune.choice([0.1, 0.25, 0.5]),  # Keep secondary
    
    "batch_size": 8  # Keep fixed for stability
}


In [15]:
analysis = tune.run(
    train_deepmreye_ray,
    config=search_space,
    metric="loss",
    mode="min",
    num_samples=30,
    raise_on_failed_trial=False,
    resources_per_trial={"cpu": 4, "gpu": 1},
)

print("Best config:", analysis.best_config)


2026-01-28 12:52:08,171	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949


Trial name,loss
train_deepmreye_ray_27ada_00000,0.107751
train_deepmreye_ray_27ada_00001,0.103677
train_deepmreye_ray_27ada_00002,0.0911188
train_deepmreye_ray_27ada_00003,0.107507
train_deepmreye_ray_27ada_00004,0.148134
train_deepmreye_ray_27ada_00005,0.142203
train_deepmreye_ray_27ada_00006,0.141198
train_deepmreye_ray_27ada_00007,0.0904994
train_deepmreye_ray_27ada_00008,0.11268
train_deepmreye_ray_27ada_00009,0.0866823


RuntimeError: Caught unexpected exception: Task was killed due to the node running low on memory.
Memory on the node (IP: 172.20.30.17, ID: a85ce7b65150ad8ec39f1b1c8aa9b76e18ec71a9deab3adfaa4ac9e0) where the lease (lease ID: 16000000a7ec6a035eed7f0e50dd3c0c588996a40c910cf53bb8dd103b6fe323, name=ImplicitFunc.__init__, pid=3606139, memory used=0.99GB) was running was 83.97GB / 88.38GB (0.950138), which exceeds the memory usage threshold of 0.95. Ray killed this worker (ID: 6cf88a2971cbcee23518b8d03d886ffa810acfe17912b5f39c86bc3c) because it was the most recently scheduled task; to see more information about memory usage on this node, use `ray logs raylet.out -ip 172.20.30.17`. To see the logs of the worker, use `ray logs worker-6cf88a2971cbcee23518b8d03d886ffa810acfe17912b5f39c86bc3c*out -ip 172.20.30.17. Top 10 memory users:
PID	MEM(GB)	COMMAND
3606139	0.99	ray::ImplicitFunc.train
3451778	0.43	/home/CemalKoba/venvs/deepmreye-gpu/bin/python -m ipykernel_launcher -f /home/CemalKoba/.local

# Single training

In [17]:
train_list_rs = np.loadtxt('/mnt/compneuro/deepmreye_finetuning/Zosia/train_list_rs.txt', dtype=str)
test_list_rs = np.loadtxt('/mnt/compneuro/deepmreye_finetuning/Zosia/test_list_rs.txt', dtype=str)

print(f"📊 Train samples: {len(train_list_rs)}")
print(f"📊 Test samples: {len(test_list_rs)}")

📊 Train samples: 23
📊 Test samples: 6


In [18]:
train_list_rs_absolute = []
for f in train_list_rs:
    if f.startswith('./processed_data/'):
        absolute_path = f.replace('./processed_data/', '/mnt/compneuro/deepmreye_finetuning/processed_data/')
    else:
        absolute_path = f
    train_list_rs_absolute.append(absolute_path)

test_list_rs_absolute = []
for f in test_list_rs:
    if f.startswith('./processed_data/'):
        absolute_path = f.replace('./processed_data/', '/mnt/compneuro/deepmreye_finetuning/processed_data/')
    else:
        absolute_path = f
    test_list_rs_absolute.append(absolute_path)

In [21]:
test_list

['/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state/9081.npz',
 '/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state/9082.npz',
 '/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state/9087.npz',
 '/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state/9023.npz',
 '/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state/9063.npz',
 '/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state/9085.npz']

In [30]:
train_list = train_list_rs[5:]   # 18
val_list   = train_list_rs[:5]   # 5
test_list = test_list_rs_absolute
generators_rs = create_holdout_generators(
    train_list=train_list, 
    test_list=val_list, 
    batch_size=opts['batch_size'], 
    augment_list=None, 
    mixed_batches=True
)


Training set (/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state) contains 18 subjects: 
['9071', '9077', '9033', '9050', '9074', '9084', '9057', '9013', '9100', '9092',
 '9091', '9001', '9029', '9007', '9055', '9053', '9008', '9041']
Test set (/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state) contains 5 subjects: 
['9088', '9070', '9083', '9048', '9025']


In [43]:
from phase1b_hybrid_training import train_hybrid_model, get_hybrid_params

opts = get_hybrid_params()
# ... create generators ...

model, model_inference, history, metrics = train_hybrid_model(
    dataset="resting_state",
    generators=generators_rs,
    opts=opts,
    num_train_subjects=18,
    num_val_subjects=6,
    timepoints_per_subject=182
)


📊 DATA CONFIGURATION
Training subjects:   18
Validation subjects: 6
Timepoints/subject:  182
Batch size:          8

Total train samples: 3276
Total val samples:   1092

Steps per epoch:     409
Validation steps:    136

Samples per epoch:   3272 (99.9% of train data)


🎯 CREATING HYBRID MODEL
Architecture: depth=4, fc=256
Regularization: dropout=0.4, noise=0.02
Smooth L1 Delta: 1.0
Loss weights - Euclidean: 1.0, Confidence: 0.25

✅ Hybrid model compiled successfully


🚀 STARTING HYBRID TRAINING
Strategy: Old capacity + Strong regularization
  • Model: depth=4, fc=256
  • Dropout: 0.4 (was 0.1)
  • Delta: 1.0 (was 0.03)
  • Data usage: 409 steps (was 60)

Epoch 1/100
409/409 [==============================] - ETA: 0s - loss: 0.4500 - smoothl1_loss: 0.4062 - confidence_loss: 0.1753
📊 Epoch 1 Summary:
   Train Loss: 0.4062
   Val Loss:   0.2931
   Gap:        0.1132
   Ratio:      0.72x
   ✅ Good generalization
409/409 [==============================] - 72s 100ms/step - loss: 0.4500 - s

In [41]:
(evaluation_ft, scores_ft) = train.evaluate_model(
    dataset='fine_tuned_resting_state', 
    model=model_inference, 
    generators=generators_rs, 
    save=False, 
    model_description='fine_tuned_on_rs', 
    verbose=2
)

📈 Evaluating fine-tuned model...
1 / 18 - Model Performance for /mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state/9071.npz
              Pearson             R^2-Score               Eucl. Error             
                    X     Y  Mean         X      Y   Mean        Mean Median   Std
Default         0.187 0.306 0.247    -0.561  0.030 -0.265       0.215  0.181 0.136
Default subTR   0.065 0.136 0.101    -0.704 -0.003 -0.354       0.271  0.196 0.241
Refined         0.187 0.306 0.247    -0.561  0.030 -0.265       0.215  0.181 0.136
Refined subTR   0.065 0.136 0.101    -0.704 -0.003 -0.354       0.271  0.196 0.241


2 / 18 - Model Performance for /mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state/9077.npz
              Pearson             R^2-Score               Eucl. Error             
                    X     Y  Mean         X      Y   Mean        Mean Median   Std
Default        -0.024 0.050 0.013    -0.140 -5.846 -2.993       0.323  0.

In [38]:
fig = analyse.visualise_predictions_slider(
    evaluation_ft, 
    scores_ft, 
    color="rgb(0, 150, 175)", 
    bg_color="rgb(255,255,255)",
    ylim=[-1, 1],
)
fig.show()

FigureWidget({
    'data': [{'boxpoints': 'all',
              'fillcolor': 'rgb(180, 180, 180)',
              'line': {'color': 'rgb(0,0,0)'},
              'marker': {'color': 'rgb(0, 150, 175)',
                         'line': {'color': 'rgb(0,0,0)', 'width': 2},
                         'opacity': 0.65,
                         'size': 12},
              'name': 'Default',
              'pointpos': 0,
              'text': [participant 9081, participant 9082, participant 9087,
                       participant 9023, participant 9063, participant 9085,
                       participant 9081, participant 9082, participant 9087,
                       participant 9023, participant 9063, participant 9085],
              'type': 'box',
              'uid': '48b7b2c3-a1d6-468b-b64b-9030ab324ee8',
              'x': [Pearson, Pearson, Pearson, Pearson, Pearson, Pearson,
                    R^2-Score, R^2-Score, R^2-Score, R^2-Score, R^2-Score,
                    R^2-Score],
         

In [2]:
ls

Add_ocular_and_lesion_info_to_stroke_df_Kasia.ipynb
datasets_1to5.h5
DeepMReye_Zosia.ipynb
HYBRID_INTEGRATION_GUIDE.py
hybrid_training_curves.png
INTEGRATION_GUIDE.py
modelinference_resting_state_hybrid.h5
modelinference_resting_state_optimized.h5
phase1b_hybrid_training.py
phase1_optimized_training.py
__pycache__/
smooth_loss.py
training_curves.png
Transfer_Learning.ipynb


In [39]:
# Check prediction range vs ground truth range
print("Predictions - min/max/mean:", pred.min(), pred.max(), pred.mean())
print("Ground truth - min/max/mean:", truth.min(), truth.max(), truth.mean())

NameError: name 'pred' is not defined

## TRANSFER

In [5]:
opts

{'kernel': 3,
 'lr': 5e-05,
 'filters': 32,
 'multiplier': 2,
 'depth': 2,
 'dropout_rate': 0.3,
 'num_dense': 2,
 'num_fc': 128,
 'gaussian_noise': 0.05,
 'activation': <function deepmreye.util.util.mish(x)>,
 'groups': 8,
 'inner_timesteps': 10,
 'loss_euclidean': 1,
 'loss_confidence': 0.5,
 'epochs': 100,
 'steps_per_epoch': 409,
 'validation_steps': 136,
 'train_test_split': 0.6,
 'batch_size': 8,
 'mixed_batches': True,
 'mc_dropout': True,
 'rotation_x': 5,
 'rotation_y': 5,
 'rotation_z': 5,
 'shift': 4,
 'zoom': 0.15,
 'smooth_l1_delta': 1}

In [6]:
def create_holdout_generators(datasets=None, train_split=0.6, train_list=None, test_list=None, **args):
    # If train_list and test_list are provided, use them directly
    if train_list is not None and test_list is not None:
        full_training_list = train_list
        full_testing_list = test_list
    else:
        # Otherwise, build from datasets
        full_training_list, full_testing_list = list(), list()
        for fn_data in datasets:
            this_file_list = [fn_data + p for p in os.listdir(fn_data)]
            np.random.shuffle(this_file_list)
            split_idx = int(train_split * len(this_file_list))
            this_training_list = this_file_list[0:split_idx]
            this_testing_list = this_file_list[split_idx:]
            full_training_list.extend(this_training_list)
            full_testing_list.extend(this_testing_list)

    # Call create_generators with the final lists
    (
        training_generator,
        testing_generator,
        single_testing_generators,
        single_testing_names,
        single_training_generators,
        single_training_names,
    ) = data_generator.create_generators(full_training_list, full_testing_list, **args)

    return (
        training_generator,
        testing_generator,
        single_testing_generators,
        single_testing_names,
        single_training_generators,
        single_training_names,
        full_testing_list,
        full_training_list,
    )

In [11]:
train_list_rs=np.loadtxt('/mnt/compneuro/deepmreye_finetuning/Zosia/train_list_rs.txt',dtype=str)
test_list_rs=np.loadtxt('/mnt/compneuro/deepmreye_finetuning/Zosia/test_list_rs.txt',dtype=str)

# Load the trained model 
generators = data_generator.create_generators(train_list_rs,
                                              train_list_rs)
generators = (*generators, train_list_rs, train_list_rs
              )  
opts = model_opts.get_opts()


(model, model_inference) = train.train_model(dataset="prediction",
                                             generators=generators,
                                             opts=opts,
                                             return_untrained=True)
model_inference.load_weights('/mnt/compneuro/Neuronus/datasets_1to5.h5')



Training set (/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state) contains 23 subjects: 
['9088', '9070', '9083', '9048', '9025', '9071', '9077', '9033', '9050', '9074',
 '9084', '9057', '9013', '9100', '9092', '9091', '9001', '9029', '9007', '9055',
 '9053', '9008', '9041']
Test set (/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state) contains 23 subjects: 
['9088', '9070', '9083', '9048', '9025', '9071', '9077', '9033', '9050', '9074',
 '9084', '9057', '9013', '9100', '9092', '9091', '9001', '9029', '9007', '9055',
 '9053', '9008', '9041']


In [9]:
# Freeze all layers up to and including 'flatten'
for layer in model.layers:
    if 'conv' in layer.name or 'pooling' in layer.name or 'flatten' in layer.name:
        layer.trainable = False

In [16]:
generators = create_holdout_generators(
            train_list=train_list,
            test_list=test_list,
            batch_size=opts['batch_size'],
            augment_list=(
                (opts['rotation_x'], opts['rotation_y'], opts['rotation_z']),
                opts['shift'],
                opts['zoom']
            ),
            mixed_batches=True
        )

Training set (/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state) contains 18 subjects: 
['9071', '9077', '9033', '9050', '9074', '9084', '9057', '9013', '9100', '9092',
 '9091', '9001', '9029', '9007', '9055', '9053', '9008', '9041']
Test set (/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state) contains 6 subjects: 
['9081', '9082', '9087', '9023', '9063', '9085']


In [15]:
train_list = train_list_rs[5:]   # 18
val_list   = train_list_rs[:5]   # 5
test_list = test_list_rs
generators_rs = create_holdout_generators(
    train_list=train_list, 
    test_list=val_list, 
    batch_size=opts['batch_size'], 
    augment_list=None, 
    mixed_batches=True
)

Training set (/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state) contains 18 subjects: 
['9071', '9077', '9033', '9050', '9074', '9084', '9057', '9013', '9100', '9092',
 '9091', '9001', '9029', '9007', '9055', '9053', '9008', '9041']
Test set (/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state) contains 5 subjects: 
['9088', '9070', '9083', '9048', '9025']


In [17]:
import platform
from os.path import join
import numpy as np
import tensorflow as tf
import tensorflow.keras.backend as K
from deepmreye import architecture
from deepmreye.util import data_generator, util
from tensorflow.keras.callbacks import EarlyStopping, Callback, ReduceLROnPlateau
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
import os 
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async" 
import tensorflow as tf 
tf.keras.backend.clear_session() 
import pandas as pd 
from deepmreye import analyse, architecture, preprocess, train 
from deepmreye.util import data_generator, model_opts, util 
import numpy as np 

gpus = tf.config.experimental.list_physical_devices('GPU') 
tf.config.experimental.set_memory_growth(gpus[0], True)

def get_adam_optimizer(learning_rate):
    is_mac = platform.system() == "Darwin"
    is_arm = platform.machine() in ["arm64", "aarch64"]

    if is_mac and is_arm:
        from keras.optimizers.legacy import Adam
        print("Apple Silicon detected - using legacy Adam optimizer.")
    else:
        from keras.optimizers import Adam
    return Adam(learning_rate=learning_rate)


class TransferLearningLogger(Callback):
    """Enhanced callback for transfer learning with degree-based metrics"""
    
    def __init__(self, test_subject_id):
        super().__init__()
        self.test_subject = test_subject_id
        self.train_losses = []
        self.val_losses = []
        self.epochs_no_improve = 0
        self.best_val_loss = float('inf')
        
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        
        train_loss = logs.get('smoothl1_loss', 0)
        val_loss = logs.get('val_smoothl1_loss', 0)
        
        self.train_losses.append(train_loss)
        self.val_losses.append(val_loss)
        
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            self.epochs_no_improve = 0
        else:
            self.epochs_no_improve += 1
        
        overfit_ratio = val_loss / train_loss if train_loss > 0 else 1.0
        
        # Estimate MAE in degrees (assuming smooth_l1 loss ≈ MAE for small errors)
        train_mae_deg = train_loss
        val_mae_deg = val_loss
        
        print(f"\n📊 Subject {self.test_subject} - Epoch {epoch + 1}:")
        print(f"   Train Loss: {train_loss:.4f} (~{train_mae_deg:.2f}°)")
        print(f"   Val Loss:   {val_loss:.4f} (~{val_mae_deg:.2f}°)")
        print(f"   Ratio:      {overfit_ratio:.2f}x")
        
        if overfit_ratio > 2.0:
            print(f"   🔴 Overfitting (val {overfit_ratio:.2f}x train)")
        elif overfit_ratio > 1.5:
            print(f"   🟡 Slight overfitting")
        else:
            print(f"   ✅ Good generalization")
    
    def plot_history(self, save_path):
        """Plot training curves for this fold"""
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        epochs = range(1, len(self.train_losses) + 1)
        
        # Loss plot
        axes[0].plot(epochs, self.train_losses, 'b-', label='Train Loss', linewidth=2)
        axes[0].plot(epochs, self.val_losses, 'r-', label='Val Loss', linewidth=2)
        axes[0].set_xlabel('Epoch', fontsize=12)
        axes[0].set_ylabel('Smooth L1 Loss (≈ degrees)', fontsize=12)
        axes[0].set_title(f'Subject {self.test_subject} - Training Curves', fontsize=14, fontweight='bold')
        axes[0].legend(fontsize=11)
        axes[0].grid(True, alpha=0.3)
        
        # Overfitting ratio plot
        ratios = [v/t if t > 0 else 1.0 for v, t in zip(self.val_losses, self.train_losses)]
        axes[1].plot(epochs, ratios, 'g-', linewidth=2)
        axes[1].axhline(y=1.0, color='gray', linestyle='--', label='Perfect (1.0x)')
        axes[1].axhline(y=1.5, color='orange', linestyle='--', label='Warning (1.5x)')
        axes[1].axhline(y=2.0, color='red', linestyle='--', label='Severe (2.0x)')
        axes[1].set_xlabel('Epoch', fontsize=12)
        axes[1].set_ylabel('Val/Train Ratio', fontsize=12)
        axes[1].set_title('Overfitting Monitor', fontsize=14, fontweight='bold')
        axes[1].legend(fontsize=10)
        axes[1].grid(True, alpha=0.3)
        axes[1].set_ylim([0, min(max(ratios) * 1.1, 5.0)])
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=150)
        plt.close()


def get_transfer_learning_opts():
    """
    Optimized parameters for transfer learning fine-tuning
    """
    opts = {
        # Architecture (must match pretrained model)
        'kernel': 3,
        'filters': 32,
        'multiplier': 2,
        'depth': 4,
        'num_dense': 2,
        'num_fc': 1024,  # Match pretrained
        'groups': 8,
        'activation': 'mish',
        'inner_timesteps': 10,
        
        # Fine-tuning specific
        'lr': 1e-6,              # 🔥 Lower LR for fine-tuning (was 2e-5)
        'dropout_rate': 0.4,     # 🔥 High dropout (pretrained had 0.1)
        'gaussian_noise': 0,     # Keep 0 to not disturb pretrained features
        'mc_dropout': True,
        
        # Loss function
        'smooth_l1_delta': 1.0,  # Match resting state movement range
        'loss_euclidean': 1.0,
        'loss_confidence': 0.1,  # Match pretrained (was 0.1)
        
        # Training
        'epochs': 100,            # Fewer epochs for fine-tuning
        'batch_size': 8,
        'mixed_batches': True,
        
        # Data steps (will be calculated per fold)
        'steps_per_epoch': None,
        'validation_steps': None,
        'train_test_split': 0.6,
        
        # Augmentation - REDUCED (don't disturb pretrained features too much)
        'rotation_x': 2,         # Reduced from 5
        'rotation_y': 2,
        'rotation_z': 2,
        'shift': 2,              # Reduced from 4
        'zoom': 0.05,            # Reduced from 0.15
    }
    
    return opts


def freeze_backbone(model, strategy='backbone_only'):
    """
    Freeze layers for transfer learning
    
    Args:
        model: The training model
        strategy: 'backbone_only' or 'aggressive'
    
    Returns:
        Number of frozen layers
    """
    frozen_count = 0
    
    print("\n" + "="*60)
    print("🧊 FREEZING LAYERS FOR TRANSFER LEARNING")
    print("="*60)
    print(f"Strategy: {strategy}")
    print()
    
    if strategy == 'backbone_only':
        # Freeze all convolutional backbone (up to flatten)
        freeze_keywords = ['conv3d', 'pooling', 'group_normalization', 
                          'activation', 'add', 'gaussian_noise', 'flatten']
        
        for layer in model.layers:
            if any(keyword in layer.name for keyword in freeze_keywords):
                layer.trainable = False
                frozen_count += 1
                print(f"  ❄️  Frozen: {layer.name}")
    
    elif strategy == 'aggressive':
        # Freeze backbone + first dense layer
        freeze_keywords = ['conv3d', 'pooling', 'group_normalization', 
                          'activation', 'add', 'gaussian_noise', 'flatten',
                          'repeat_vector', 'lambda', 'dense_30']  # confidence head
        
        for layer in model.layers:
            layer_name = layer.name
            if any(keyword in layer_name for keyword in freeze_keywords):
                layer.trainable = False
                frozen_count += 1
                print(f"  ❄️  Frozen: {layer_name}")
            # Also freeze first dense in each branch (dense, dense_3, dense_6, etc.)
            elif layer_name.startswith('dense') and not layer_name.startswith('dense_1'):
                if '_' not in layer_name or int(layer_name.split('_')[1]) % 3 == 0:
                    layer.trainable = False
                    frozen_count += 1
                    print(f"  ❄️  Frozen: {layer_name}")
    
    print()
    print(f"Total frozen layers: {frozen_count}")
    print(f"Total trainable params: {sum([K.count_params(w) for w in model.trainable_weights]):,}")
    print("="*60 + "\n")
    
    return frozen_count


def calculate_loso_steps(num_subjects, timepoints_per_subject, batch_size):
    """Calculate steps for LOSO (Leave-One-Subject-Out)"""
    train_subjects = num_subjects - 1
    test_subjects = 1
    
    train_samples = train_subjects * timepoints_per_subject
    test_samples = test_subjects * timepoints_per_subject
    
    steps_per_epoch = train_samples // batch_size
    validation_steps = test_samples // batch_size
    
    return steps_per_epoch, validation_steps


def create_model_with_smoothl1(input_shape, opts):
    """Create model with Smooth L1 loss (same as hybrid)"""
    
    base_model, model_inference = architecture.create_standard_model(
        input_shape,
        opts
    )
    
    def smooth_l1(y_true, y_pred):
        delta = opts.get("smooth_l1_delta", 1.0)
        error = y_true - y_pred
        abs_error = tf.abs(error)
        return tf.where(
            abs_error < delta,
            0.5 * tf.square(error),
            delta * (abs_error - 0.5 * delta)
        )
    
    inputs = base_model.inputs
    out_regression = model_inference.outputs[0]
    out_confidence = model_inference.outputs[1]
    real_regression = inputs[1]
    
    euclidean_loss = tf.sqrt(tf.reduce_sum(
        smooth_l1(real_regression, out_regression),
        axis=-1
    ))
    
    confidence_loss = tf.square(euclidean_loss - out_confidence)
    
    train_model = tf.keras.Model(
        inputs=inputs,
        outputs=[],
        name="transfer_learning_model"
    )
    
    train_model.add_loss(opts["loss_euclidean"] * tf.reduce_mean(euclidean_loss))
    train_model.add_loss(opts["loss_confidence"] * tf.reduce_mean(confidence_loss))
    
    train_model.add_metric(tf.reduce_mean(euclidean_loss), name="smoothl1_loss")
    train_model.add_metric(tf.reduce_mean(confidence_loss), name="confidence_loss")
    
    optimizer = get_adam_optimizer(opts["lr"])
    optimizer.clipnorm = 1.0
    
    train_model.compile(optimizer=optimizer)
    
    return train_model, model_inference


def train_single_fold(
    fold_idx,
    test_subject,
    train_subjects,
    generators,
    opts,
    pretrained_weights_path,
    freeze_strategy='backbone_only',
    save_dir="./transfer_learning_results",
    timepoints_per_subject=182,
    verbose=1
):
    """
    Train a single LOSO fold
    
    Args:
        fold_idx: Fold number (0-28 for 29 subjects)
        test_subject: Subject ID being held out
        train_subjects: List of training subject IDs
        generators: Data generators
        opts: Training options
        pretrained_weights_path: Path to datasets_1to5.h5
        freeze_strategy: 'backbone_only' or 'aggressive'
        save_dir: Directory to save results
        timepoints_per_subject: Number of timepoints per subject
        verbose: Verbosity level
    
    Returns:
        Dictionary with results
    """
    
    print("\n" + "="*70)
    print(f"FOLD {fold_idx + 1}/29 - TESTING ON SUBJECT {test_subject}")
    print("="*70)
    
    # Clear session
    K.clear_session()
    
    # Calculate steps
    num_train = len(train_subjects)
    steps_per_epoch, validation_steps = calculate_loso_steps(
        num_subjects=29,
        timepoints_per_subject=timepoints_per_subject,
        batch_size=opts['batch_size']
    )
    opts['steps_per_epoch'] = steps_per_epoch
    opts['validation_steps'] = validation_steps
    
    print(f"\nData split:")
    print(f"  Training subjects: {num_train}")
    print(f"  Test subject: {test_subject}")
    print(f"  Steps per epoch: {steps_per_epoch}")
    print(f"  Validation steps: {validation_steps}")
    
    # Unpack generators
    (training_generator, testing_generator, _, _, _, _, _, _) = generators
    
    # Get input shape
    ((X, y), _) = next(training_generator)
    
    # Create model
    print("\n📦 Creating model...")
    model, model_inference = create_model_with_smoothl1(X.shape[1::], opts)
    
    # Load pretrained weights
    print(f"\n🔄 Loading pretrained weights from: {pretrained_weights_path}")
    try:
        model_inference.load_weights(pretrained_weights_path)
        print("✅ Pretrained weights loaded successfully")
    except Exception as e:
        print(f"❌ Error loading weights: {e}")
        print("⚠️  Continuing without pretrained weights...")
    
    # Freeze backbone
    freeze_backbone(model, strategy=freeze_strategy)
    
    # Recompile after freezing
    optimizer = get_adam_optimizer(opts["lr"])
    optimizer.clipnorm = 1.0
    model.compile(optimizer=optimizer)
    
    # Setup callbacks
    lr_sched = util.step_decay_schedule(
        initial_lr=opts["lr"],
        decay_factor=0.95,
        num_epochs=opts["epochs"]
    )
    
    early_stop_cb = EarlyStopping(
        monitor='val_smoothl1_loss',
        patience=15,  # Lower patience for fine-tuning
        restore_best_weights=True,
        verbose=1,
        mode='min'
    )
    
    reduce_lr_cb = ReduceLROnPlateau(
        monitor='val_smoothl1_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        verbose=1
    )
    
    logger = TransferLearningLogger(test_subject_id=test_subject)
    
    callbacks_list = [lr_sched, early_stop_cb, reduce_lr_cb, logger]
    
    # Train
    print("\n🚀 Starting fine-tuning...")
    history = model.fit(
        training_generator,
        steps_per_epoch=opts["steps_per_epoch"],
        epochs=opts["epochs"],
        validation_data=testing_generator,
        validation_steps=opts["validation_steps"],
        callbacks=callbacks_list,
        workers=1,
        use_multiprocessing=False,
        verbose=verbose
    )
    
    # Save results
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    
    # Save training curves
    curve_path = join(save_dir, f'fold_{fold_idx:02d}_subject_{test_subject}_curves.png')
    logger.plot_history(curve_path)
    
    # Save model weights
    weights_path = join(save_dir, f'fold_{fold_idx:02d}_subject_{test_subject}_weights.h5')
    model_inference.save_weights(weights_path)
    
    # Collect results
    results = {
        'fold_idx': fold_idx,
        'test_subject': test_subject,
        'train_subjects': train_subjects,
        'train_losses': logger.train_losses,
        'val_losses': logger.val_losses,
        'best_val_loss': logger.best_val_loss,
        'final_train_loss': logger.train_losses[-1] if logger.train_losses else None,
        'final_val_loss': logger.val_losses[-1] if logger.val_losses else None,
        'weights_path': weights_path,
        'curves_path': curve_path
    }
    
    # Print summary
    print("\n" + "="*70)
    print(f"FOLD {fold_idx + 1} COMPLETE - Subject {test_subject}")
    print("="*70)
    print(f"Best Val Loss: {results['best_val_loss']:.4f}° (approx)")
    print(f"Final Train Loss: {results['final_train_loss']:.4f}°")
    print(f"Final Val Loss: {results['final_val_loss']:.4f}°")
    if results['final_train_loss'] and results['final_val_loss']:
        ratio = results['final_val_loss'] / results['final_train_loss']
        print(f"Overfit Ratio: {ratio:.2f}x")
    print("="*70 + "\n")
    
    return results


def run_loso_transfer_learning(
    subject_list,
    data_path,
    pretrained_weights_path,
    opts=None,
    freeze_strategy='backbone_only',
    save_dir="./transfer_learning_results",
    timepoints_per_subject=182,
    start_fold=0,
    end_fold=None
):
    """
    Run full LOSO cross-validation with transfer learning
    
    Args:
        subject_list: List of all subject IDs
        data_path: Path to preprocessed data directory
        pretrained_weights_path: Path to datasets_1to5.h5
        opts: Training options (if None, uses defaults)
        freeze_strategy: 'backbone_only' or 'aggressive'
        save_dir: Directory to save results
        timepoints_per_subject: Timepoints per subject
        start_fold: Start from this fold (for resuming)
        end_fold: End at this fold (for partial runs)
    
    Returns:
        List of results dictionaries for each fold
    """
    
    if opts is None:
        opts = get_transfer_learning_opts()
    
    if end_fold is None:
        end_fold = len(subject_list)
    
    all_results = []
    
    print("\n" + "="*70)
    print("LOSO TRANSFER LEARNING - CONFIGURATION")
    print("="*70)
    print(f"Total subjects: {len(subject_list)}")
    print(f"Folds to run: {start_fold + 1} to {end_fold}")
    print(f"Pretrained weights: {pretrained_weights_path}")
    print(f"Freeze strategy: {freeze_strategy}")
    print(f"Learning rate: {opts['lr']}")
    print(f"Epochs per fold: {opts['epochs']}")
    print(f"Dropout: {opts['dropout_rate']}")
    print(f"Smooth L1 delta: {opts['smooth_l1_delta']}")
    print("="*70 + "\n")
    
    for fold_idx in range(start_fold, end_fold):
        test_subject = subject_list[fold_idx]
        train_subjects = [s for s in subject_list if s != test_subject]
        
        # Create subject lists for this fold
        train_list = [join(data_path, f"{s}.npz") for s in train_subjects]
        test_list = [join(data_path, f"{test_subject}.npz")]
        
        # Create generators
        generators = create_holdout_generators(
            train_list=train_list,
            test_list=test_list,
            batch_size=opts['batch_size'],
            augment_list=(
                (opts['rotation_x'], opts['rotation_y'], opts['rotation_z']),
                opts['shift'],
                opts['zoom']
            ),
            mixed_batches=True
        )
        
        # Train this fold
        fold_results = train_single_fold(
            fold_idx=fold_idx,
            test_subject=test_subject,
            train_subjects=train_subjects,
            generators=generators,
            opts=opts,
            pretrained_weights_path=pretrained_weights_path,
            freeze_strategy=freeze_strategy,
            save_dir=save_dir,
            timepoints_per_subject=timepoints_per_subject,
            verbose=1
        )
        
        all_results.append(fold_results)
        
        # Save intermediate results
        results_path = join(save_dir, 'loso_results.pkl')
        with open(results_path, 'wb') as f:
            pickle.dump(all_results, f)
        print(f"💾 Intermediate results saved to: {results_path}\n")
    
    # Final summary
    print("\n" + "="*70)
    print("LOSO TRANSFER LEARNING - FINAL SUMMARY")
    print("="*70)
    
    best_losses = [r['best_val_loss'] for r in all_results]
    final_train_losses = [r['final_train_loss'] for r in all_results]
    final_val_losses = [r['final_val_loss'] for r in all_results]
    
    print(f"Completed folds: {len(all_results)}")
    print(f"\nBest validation loss across folds:")
    print(f"  Mean: {np.mean(best_losses):.4f}° (approx)")
    print(f"  Std:  {np.std(best_losses):.4f}°")
    print(f"  Min:  {np.min(best_losses):.4f}°")
    print(f"  Max:  {np.max(best_losses):.4f}°")
    
    print(f"\nFinal validation loss across folds:")
    print(f"  Mean: {np.mean(final_val_losses):.4f}°")
    print(f"  Std:  {np.std(final_val_losses):.4f}°")
    
    ratios = [v/t for v, t in zip(final_val_losses, final_train_losses)]
    print(f"\nOverfit ratios:")
    print(f"  Mean: {np.mean(ratios):.2f}x")
    print(f"  Std:  {np.std(ratios):.2f}x")
    
    print("="*70 + "\n")
    
    return all_results


# ==========================================
# USAGE EXAMPLE
# ==========================================

if __name__ == "__main__":
    """
    Example usage for LOSO transfer learning
    """
    
    # Configuration
    SUBJECT_LIST = [
        '9071', '9077', '9033', '9050', '9074', '9084', '9057', '9013', '9100', '9092',
        '9091', '9001', '9029', '9007', '9055', '9053', '9008', '9041',
        '9088', '9070', '9083', '9048', '9025', '9081', '9082', '9087', '9023', '9063', '9085',# Add your other 11 subjects
        # ... complete the list with all 29 subjects
    ]
    
    DATA_PATH = "/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state"
    PRETRAINED_WEIGHTS = "./datasets_1to5.h5"
    SAVE_DIR = "./transfer_learning_results"
    
    # Get options
    opts = get_transfer_learning_opts()
    
    # Run LOSO (can run subset of folds for testing)
    results = run_loso_transfer_learning(
        subject_list=SUBJECT_LIST,
        data_path=DATA_PATH,
        pretrained_weights_path=PRETRAINED_WEIGHTS,
        opts=opts,
        freeze_strategy='backbone_only',  # or 'aggressive'
        save_dir=SAVE_DIR,
        timepoints_per_subject=182,
        start_fold=0,   # Start from first fold
        end_fold=3   # Run all folds (or set to 3 for testing first 3)
    )
    
    print("\n✅ LOSO Transfer Learning Complete!")
    print(f"Results saved to: {SAVE_DIR}")


LOSO TRANSFER LEARNING - CONFIGURATION
Total subjects: 29
Folds to run: 1 to 3
Pretrained weights: ./datasets_1to5.h5
Freeze strategy: backbone_only
Learning rate: 1e-06
Epochs per fold: 100
Dropout: 0.4
Smooth L1 delta: 1.0

Training set (/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state) contains 28 subjects: 
['9077', '9033', '9050', '9074', '9084', '9057', '9013', '9100', '9092', '9091',
 '9001', '9029', '9007', '9055', '9053', '9008', '9041', '9088', '9070', '9083',
 '9048', '9025', '9081', '9082', '9087', '9023', '9063', '9085']
Test set (/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state) contains 1 subjects: 
['9071']

FOLD 1/29 - TESTING ON SUBJECT 9071

Data split:
  Training subjects: 28
  Test subject: 9071
  Steps per epoch: 637
  Validation steps: 22

📦 Creating model...

🔄 Loading pretrained weights from: ./datasets_1to5.h5
✅ Pretrained weights loaded successfully

🧊 FREEZING LAYERS FOR TRANSFER LEARNING
Strategy: backbone_

2026-02-04 11:24:16.551499: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907
2026-02-04 11:24:19.839144: I external/local_xla/xla/service/service.cc:168] XLA service 0x7fdd94081ec0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-04 11:24:19.839189: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA RTX A4000, Compute Capability 8.6
2026-02-04 11:24:19.855235: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1770204260.014561 1615388 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


637/637 [==============================] - ETA: 0s - loss: 1.5220 - smoothl1_loss: 1.1636 - confidence_loss: 3.5843
📊 Subject 9071 - Epoch 1:
   Train Loss: 1.1636 (~1.16°)
   Val Loss:   0.7994 (~0.80°)
   Ratio:      0.69x
   ✅ Good generalization
637/637 [==============================] - 77s 89ms/step - loss: 1.5220 - smoothl1_loss: 1.1636 - confidence_loss: 3.5843 - val_loss: 0.9607 - val_smoothl1_loss: 0.7994 - val_confidence_loss: 1.6127 - lr: 1.0000e-06
Epoch 2/100
637/637 [==============================] - ETA: 0s - loss: 1.0218 - smoothl1_loss: 0.8524 - confidence_loss: 1.6942
📊 Subject 9071 - Epoch 2:
   Train Loss: 0.8524 (~0.85°)
   Val Loss:   0.5407 (~0.54°)
   Ratio:      0.63x
   ✅ Good generalization
637/637 [==============================] - 58s 92ms/step - loss: 1.0218 - smoothl1_loss: 0.8524 - confidence_loss: 1.6942 - val_loss: 0.5947 - val_smoothl1_loss: 0.5407 - val_confidence_loss: 0.5396 - lr: 9.9050e-07
Epoch 3/100
637/637 [==============================] - E


KeyboardInterrupt



In [19]:
import platform
from os.path import join
import numpy as np
import tensorflow as tf
import tensorflow.keras.backend as K
from deepmreye import architecture
from deepmreye.util import data_generator, util
from tensorflow.keras.callbacks import EarlyStopping, Callback, ReduceLROnPlateau
import matplotlib.pyplot as plt
from pathlib import Path


def get_adam_optimizer(learning_rate):
    is_mac = platform.system() == "Darwin"
    is_arm = platform.machine() in ["arm64", "aarch64"]

    if is_mac and is_arm:
        from keras.optimizers.legacy import Adam
        print("Apple Silicon detected - using legacy Adam optimizer.")
    else:
        from keras.optimizers import Adam
    return Adam(learning_rate=learning_rate)


class TransferLearningLogger(Callback):
    """Enhanced callback for transfer learning"""
    
    def __init__(self):
        super().__init__()
        self.train_losses = []
        self.val_losses = []
        self.epochs_no_improve = 0
        self.best_val_loss = float('inf')
        
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        
        train_loss = logs.get('smoothl1_loss', 0)
        val_loss = logs.get('val_smoothl1_loss', 0)
        
        self.train_losses.append(train_loss)
        self.val_losses.append(val_loss)
        
        if val_loss < self.best_val_loss:
            self.best_val_loss = val_loss
            self.epochs_no_improve = 0
        else:
            self.epochs_no_improve += 1
        
        overfit_ratio = val_loss / train_loss if train_loss > 0 else 1.0
        
        print(f"\n📊 Epoch {epoch + 1}:")
        print(f"   Train Loss: {train_loss:.4f} (~{train_loss:.2f}°)")
        print(f"   Val Loss:   {val_loss:.4f} (~{val_loss:.2f}°)")
        print(f"   Ratio:      {overfit_ratio:.2f}x")
        
        if overfit_ratio > 2.0:
            print(f"   🔴 Overfitting (val {overfit_ratio:.2f}x train)")
        elif overfit_ratio > 1.5:
            print(f"   🟡 Slight overfitting")
        else:
            print(f"   ✅ Good generalization")
    
    def plot_history(self, save_path):
        """Plot training curves"""
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        epochs = range(1, len(self.train_losses) + 1)
        
        # Loss plot
        axes[0].plot(epochs, self.train_losses, 'b-', label='Train Loss', linewidth=2)
        axes[0].plot(epochs, self.val_losses, 'r-', label='Val Loss', linewidth=2)
        axes[0].set_xlabel('Epoch', fontsize=12)
        axes[0].set_ylabel('Smooth L1 Loss (≈ degrees)', fontsize=12)
        axes[0].set_title('Training vs Validation Loss', fontsize=14, fontweight='bold')
        axes[0].legend(fontsize=11)
        axes[0].grid(True, alpha=0.3)
        
        # Overfitting ratio plot
        ratios = [v/t if t > 0 else 1.0 for v, t in zip(self.val_losses, self.train_losses)]
        axes[1].plot(epochs, ratios, 'g-', linewidth=2)
        axes[1].axhline(y=1.0, color='gray', linestyle='--', label='Perfect (1.0x)')
        axes[1].axhline(y=1.5, color='orange', linestyle='--', label='Warning (1.5x)')
        axes[1].axhline(y=2.0, color='red', linestyle='--', label='Severe (2.0x)')
        axes[1].set_xlabel('Epoch', fontsize=12)
        axes[1].set_ylabel('Val/Train Ratio', fontsize=12)
        axes[1].set_title('Overfitting Monitor', fontsize=14, fontweight='bold')
        axes[1].legend(fontsize=10)
        axes[1].grid(True, alpha=0.3)
        axes[1].set_ylim([0, min(max(ratios) * 1.1, 5.0)])
        
        plt.tight_layout()
        plt.savefig(save_path, dpi=150)
        plt.close()


def get_transfer_learning_opts():
    """Optimized parameters for transfer learning"""
    opts = {
        'kernel': 3,
        'filters': 32,
        'multiplier': 2,
        'depth': 4,
        'num_dense': 2,
        'num_fc': 1024,
        'groups': 8,
        'activation': 'mish',
        'inner_timesteps': 10,
        
        'lr': 1e-5,
        'dropout_rate': 0.4,
        'gaussian_noise': 0,
        'mc_dropout': True,
        
        'smooth_l1_delta': 1.0,
        'loss_euclidean': 1.0,
        'loss_confidence': 0.1,
        
        'epochs': 50,
        'batch_size': 8,
        'mixed_batches': True,
        
        'steps_per_epoch': None,
        'validation_steps': None,
        'train_test_split': 0.6,
        
        'rotation_x': 2,
        'rotation_y': 2,
        'rotation_z': 2,
        'shift': 2,
        'zoom': 0.05,
    }
    return opts


def freeze_backbone(model, strategy='backbone_only'):
    """Freeze layers for transfer learning"""
    frozen_count = 0
    
    print("\n" + "="*60)
    print("🧊 FREEZING LAYERS FOR TRANSFER LEARNING")
    print("="*60)
    print(f"Strategy: {strategy}")
    print()
    
    if strategy == 'backbone_only':
        freeze_keywords = ['conv3d', 'pooling', 'group_normalization', 
                          'activation', 'add', 'gaussian_noise', 'flatten']
        
        for layer in model.layers:
            if any(keyword in layer.name for keyword in freeze_keywords):
                layer.trainable = False
                frozen_count += 1
    
    elif strategy == 'aggressive':
        freeze_keywords = ['conv3d', 'pooling', 'group_normalization', 
                          'activation', 'add', 'gaussian_noise', 'flatten',
                          'repeat_vector', 'lambda', 'dense_30']
        
        for layer in model.layers:
            layer_name = layer.name
            if any(keyword in layer_name for keyword in freeze_keywords):
                layer.trainable = False
                frozen_count += 1
            elif layer_name.startswith('dense') and not layer_name.startswith('dense_1'):
                if '_' not in layer_name or int(layer_name.split('_')[1]) % 3 == 0:
                    layer.trainable = False
                    frozen_count += 1
    
    print(f"Total frozen layers: {frozen_count}")
    print(f"Total trainable params: {sum([K.count_params(w) for w in model.trainable_weights]):,}")
    print("="*60 + "\n")
    
    return frozen_count


def create_model_with_smoothl1(input_shape, opts):
    """Create model with Smooth L1 loss"""
    
    base_model, model_inference = architecture.create_standard_model(
        input_shape,
        opts
    )
    
    def smooth_l1(y_true, y_pred):
        delta = opts.get("smooth_l1_delta", 1.0)
        error = y_true - y_pred
        abs_error = tf.abs(error)
        return tf.where(
            abs_error < delta,
            0.5 * tf.square(error),
            delta * (abs_error - 0.5 * delta)
        )
    
    inputs = base_model.inputs
    out_regression = model_inference.outputs[0]
    out_confidence = model_inference.outputs[1]
    real_regression = inputs[1]
    
    euclidean_loss = tf.sqrt(tf.reduce_sum(
        smooth_l1(real_regression, out_regression),
        axis=-1
    ))
    
    confidence_loss = tf.square(euclidean_loss - out_confidence)
    
    train_model = tf.keras.Model(
        inputs=inputs,
        outputs=[],
        name="transfer_learning_model"
    )
    
    train_model.add_loss(opts["loss_euclidean"] * tf.reduce_mean(euclidean_loss))
    train_model.add_loss(opts["loss_confidence"] * tf.reduce_mean(confidence_loss))
    
    train_model.add_metric(tf.reduce_mean(euclidean_loss), name="smoothl1_loss")
    train_model.add_metric(tf.reduce_mean(confidence_loss), name="confidence_loss")
    
    optimizer = get_adam_optimizer(opts["lr"])
    optimizer.clipnorm = 1.0
    
    train_model.compile(optimizer=optimizer)
    
    return train_model, model_inference


def train_transfer_learning(
    train_list,
    test_list,
    pretrained_weights_path,
    opts=None,
    freeze_strategy='backbone_only',
    save_dir="./transfer_learning_results",
    model_name="transfer_model"
):
    """
    Simple train/test split transfer learning
    
    Args:
        train_list: List of training subject file paths
        test_list: List of test subject file paths
        pretrained_weights_path: Path to datasets_1to5.h5
        opts: Training options
        freeze_strategy: 'backbone_only' or 'aggressive'
        save_dir: Directory to save results
        model_name: Name for saved model
    """
    
    if opts is None:
        opts = get_transfer_learning_opts()
    
    print("\n" + "="*70)
    print("TRANSFER LEARNING - SIMPLE TRAIN/TEST SPLIT")
    print("="*70)
    print(f"Training subjects: {len(train_list)}")
    print(f"Test subjects: {len(test_list)}")
    print(f"Pretrained weights: {pretrained_weights_path}")
    print(f"Freeze strategy: {freeze_strategy}")
    print("="*70 + "\n")
    
    # Clear session
    K.clear_session()
    
    # Create generators
    print("📦 Creating data generators...")
    generators = create_holdout_generators(
        train_list=train_list,
        test_list=test_list,
        batch_size=opts['batch_size'],
        augment_list=(
            (opts['rotation_x'], opts['rotation_y'], opts['rotation_z']),
            opts['shift'],
            opts['zoom']
        ),
        mixed_batches=True
    )
    
    (training_generator, testing_generator, _, _, _, _, _, _) = generators
    
    # Calculate steps
    n_train = len(train_list)
    n_test = len(test_list)
    timepoints_per_subject = 182
    
    steps_per_epoch = (n_train * timepoints_per_subject) // opts['batch_size']
    validation_steps = (n_test * timepoints_per_subject) // opts['batch_size']
    
    opts['steps_per_epoch'] = steps_per_epoch
    opts['validation_steps'] = validation_steps
    
    print(f"\nSteps per epoch: {steps_per_epoch}")
    print(f"Validation steps: {validation_steps}\n")
    
    # Get input shape
    ((X, y), _) = next(training_generator)
    
    # Create model
    print("📦 Creating model...")
    model, model_inference = create_model_with_smoothl1(X.shape[1::], opts)
    
    # Load pretrained weights
    print(f"\n🔄 Loading pretrained weights from: {pretrained_weights_path}")
    try:
        model_inference.load_weights(pretrained_weights_path)
        print("✅ Pretrained weights loaded successfully")
    except Exception as e:
        print(f"❌ Error loading weights: {e}")
        print("⚠️  Training from scratch...")
    
    # Freeze backbone
    freeze_backbone(model, strategy=freeze_strategy)
    
    # Recompile
    optimizer = get_adam_optimizer(opts["lr"])
    optimizer.clipnorm = 1.0
    model.compile(optimizer=optimizer)
    
    # Setup callbacks
    lr_sched = util.step_decay_schedule(
        initial_lr=opts["lr"],
        decay_factor=0.95,
        num_epochs=opts["epochs"]
    )
    
    early_stop_cb = EarlyStopping(
        monitor='val_smoothl1_loss',
        patience=20,
        restore_best_weights=True,
        verbose=1,
        mode='min'
    )
    
    reduce_lr_cb = ReduceLROnPlateau(
        monitor='val_smoothl1_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-7,
        verbose=1
    )
    
    logger = TransferLearningLogger()
    
    callbacks_list = [lr_sched, early_stop_cb, reduce_lr_cb, logger]
    
    # Train
    print("\n🚀 Starting training...")
    history = model.fit(
        training_generator,
        steps_per_epoch=opts["steps_per_epoch"],
        epochs=opts["epochs"],
        validation_data=testing_generator,
        validation_steps=opts["validation_steps"],
        callbacks=callbacks_list,
        workers=1,
        use_multiprocessing=False,
        verbose=1
    )
    
    # Save results
    Path(save_dir).mkdir(parents=True, exist_ok=True)
    
    # Save training curves
    curve_path = join(save_dir, f'{model_name}_training_curves.png')
    logger.plot_history(curve_path)
    print(f"\n📊 Training curves saved to: {curve_path}")
    
    # Save model weights
    weights_path = join(save_dir, f'{model_name}_weights.h5')
    model_inference.save_weights(weights_path)
    print(f"💾 Model weights saved to: {weights_path}")
    
    # Final summary
    print("\n" + "="*70)
    print("TRAINING COMPLETE")
    print("="*70)
    final_train_loss = logger.train_losses[-1]
    final_val_loss = logger.val_losses[-1]
    best_val_loss = min(logger.val_losses)
    best_epoch = logger.val_losses.index(best_val_loss) + 1
    final_ratio = final_val_loss / final_train_loss if final_train_loss > 0 else 1.0
    
    print(f"Best Val Loss:       {best_val_loss:.4f}° (epoch {best_epoch})")
    print(f"Final Train Loss:    {final_train_loss:.4f}°")
    print(f"Final Val Loss:      {final_val_loss:.4f}°")
    print(f"Train/Val Ratio:     {final_ratio:.2f}x")
    
    if final_ratio < 1.3:
        print("\n✅ EXCELLENT: Model generalizes very well!")
    elif final_ratio < 1.8:
        print("\n🟢 GOOD: Reasonable generalization")
    elif final_ratio < 2.5:
        print("\n🟡 MODERATE: Some overfitting")
    else:
        print("\n🔴 POOR: Severe overfitting")
    
    print("="*70 + "\n")
    
    return model, model_inference, history, logger


# ==========================================
# USAGE EXAMPLE
# ==========================================

if __name__ == "__main__":
    """
    Simple train/test split transfer learning
    """
    
    # Configuration
    DATA_PATH = "/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state"
    PRETRAINED_WEIGHTS = "./datasets_1to5.h5"
    SAVE_DIR = "./transfer_learning_results"
    
    # Define train/test split
    # Example: 18 train, 11 test (or adjust as needed)
    ALL_SUBJECTS = [
        '9071', '9077', '9033', '9050', '9074', '9084', '9057', '9013', '9100', '9092',
        '9091', '9001', '9029', '9007', '9055', '9053', '9008', '9041',
        '9088', '9070', '9083', '9048', '9025', '9081', '9082', '9087', '9023', '9063', '9085',# Add your other 11 subjects
        # ... complete the list with all 29 subjects
    ]
    
    # Split: first 18 for train, rest for test
    TRAIN_SUBJECTS = ALL_SUBJECTS[:18]
    TEST_SUBJECTS = ALL_SUBJECTS[18:]
    
    train_list = [join(DATA_PATH, f"{s}.npz") for s in TRAIN_SUBJECTS]
    test_list = [join(DATA_PATH, f"{s}.npz") for s in TEST_SUBJECTS]
    
    print(f"Training on {len(train_list)} subjects: {TRAIN_SUBJECTS}")
    print(f"Testing on {len(test_list)} subjects: {TEST_SUBJECTS}")
    
    # Get options
    opts = get_transfer_learning_opts()
    
    # Optional: adjust parameters
    # opts['lr'] = 5e-6  # Lower learning rate
    # opts['epochs'] = 100  # More epochs
    
    # Train
    model, model_inference, history, logger = train_transfer_learning(
        train_list=train_list,
        test_list=test_list,
        pretrained_weights_path=PRETRAINED_WEIGHTS,
        opts=opts,
        freeze_strategy='backbone_only',  # or 'aggressive'
        save_dir=SAVE_DIR,
        model_name='transfer_model_simple'
    )
    
    print("\n✅ Training complete!")
    print(f"\nModel saved to: {SAVE_DIR}")
    print("\nNext step: Run evaluate_predictions_detailed.py to see results")

Training on 18 subjects: ['9071', '9077', '9033', '9050', '9074', '9084', '9057', '9013', '9100', '9092', '9091', '9001', '9029', '9007', '9055', '9053', '9008', '9041']
Testing on 11 subjects: ['9088', '9070', '9083', '9048', '9025', '9081', '9082', '9087', '9023', '9063', '9085']

TRANSFER LEARNING - SIMPLE TRAIN/TEST SPLIT
Training subjects: 18
Test subjects: 11
Pretrained weights: ./datasets_1to5.h5
Freeze strategy: backbone_only

📦 Creating data generators...
Training set (/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state) contains 18 subjects: 
['9071', '9077', '9033', '9050', '9074', '9084', '9057', '9013', '9100', '9092',
 '9091', '9001', '9029', '9007', '9055', '9053', '9008', '9041']
Test set (/mnt/compneuro/deepmreye_finetuning/processed_data/dataset7_resting_state) contains 11 subjects: 
['9088', '9070', '9083', '9048', '9025', '9081', '9082', '9087', '9023', '9063',
 '9085']

Steps per epoch: 409
Validation steps: 250

📦 Creating model...

🔄 Loading